# Convolutional Neural Network II: Classification d'objets Recyclables ou Non

In [ ]:
# !rm -rf ./DATASET
!rm -rf ./dataset


In [8]:
from google.colab import drive
drive.mount('/gdrive')


ModuleNotFoundError: No module named 'google.colab'

In [ ]:
!ls '/gdrive/MyDrive/data'


In [ ]:
fichier_archive = 'gdrive/MyDrive/data/archive.zip'
# Origine du dataset: https://www.kaggle.com/datasets/techsash/waste-classification-data?resource=download

# !unzip -o '/{fichier_archive}' -d './' # <= Activer pour vraiment importer


In [ ]:
!rm -rf ./dataset


In [ ]:
import os


In [ ]:
data_dir = "DATASET"


In [ ]:
train_dir = os.path.join(data_dir, "TRAIN")
test_dir = os.path.join(data_dir, "TEST")


In [ ]:
os.listdir(train_dir)


In [ ]:
train_r_dir = os.path.join(train_dir, "R")
train_o_dir = os.path.join(train_dir, "O")


In [ ]:
from tabulate import tabulate
import os

train_r_count = len(os.listdir(train_r_dir))
train_o_count = len(os.listdir(train_o_dir))
test_r_count = len(os.listdir(os.path.join(test_dir, "R")))
test_o_count = len(os.listdir(os.path.join(test_dir, "O")))

total_r = train_r_count + test_r_count
total_o = train_o_count + test_o_count
total_train = train_r_count + train_o_count
total_test = test_r_count + test_o_count
grand_total = total_train + total_test


data = [
    ["Recyclable (R)", train_r_count, test_r_count, total_r],
    ["Organic (O)", train_o_count, test_o_count, total_o],
    ["Cumulative", total_train, total_test, grand_total]
]

headers = ["Category", "Train", "Test", "Cumulative"]

print(tabulate(data, headers=headers, tablefmt="grid"))


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg


In [ ]:
r_index = 568
o_index = 1596

recyclable_imgs = [os.path.join(train_r_dir, filename) for filename in os.listdir(train_r_dir)]
organic_imgs = [os.path.join(train_o_dir, filename) for filename in os.listdir(train_o_dir)]

img_r = mpimg.imread(recyclable_imgs[r_index])
img_o = mpimg.imread(organic_imgs[o_index])


In [ ]:
plt.imshow(img_r)


In [ ]:
plt.imshow(img_o)


In [ ]:
# training_labels.shape # Inconnu ! → Solution = ImageDataGenerator


## ImageDataGenerator

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator


In [ ]:
train_data_generator = ImageDataGenerator(rescale=1/255.0)
test_data_generator = ImageDataGenerator(rescale=1/255.0)


In [ ]:
train_generator = train_data_generator.flow_from_directory(
    directory = train_dir,
    target_size = (224, 224),
    batch_size=32,
    class_mode="binary"
)

test_generator = test_data_generator.flow_from_directory(
    directory = test_dir,
    target_size = (224, 224),
    batch_size=32,
    class_mode="binary"
)


In [ ]:
batch_images, batch_labels = next(iter(train_generator))


In [ ]:
batch_images.shape


In [ ]:
batch_labels.shape, batch_labels


In [ ]:
train_generator.class_indices


In [ ]:
labels = list(train_generator.class_indices.keys())
labels


In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt

index = 30

print(f"Image # {index} - Label:", labels[int(batch_labels[index])])

# Create a figure and axes with a size that makes the image appear around 112x112
# The exact figsize might need some trial and error depending on your display
plt.figure(figsize=(2, 2)) # Adjust figsize as needed to get the desired visual size

# Display the original size image from the batch
img_to_display = batch_images[index]
plt.imshow(img_to_display)

# Turn off the axis
plt.axis('off')

plt.show()


In [ ]:
import random
import tensorflow as tf
import matplotlib.pyplot as plt

nb_imgs = 16 # Max 32 car batch en contient que 32
random_indices = random.sample(range(0, batch_images.shape[0]), nb_imgs) # Get 10 random indices from the batch (content 32)

# Determine the number of rows and columns for the subplots
n_images = len(random_indices)
n_cols = 8 # You can adjust the number of columns as needed
n_rows = (n_images + n_cols - 1) // n_cols

plt.figure(figsize=(n_cols * 1.5, n_rows * 1.5)) # Adjust figure size for smaller visual display

for i, img_index in enumerate(random_indices):
  plt.subplot(n_rows, n_cols, i + 1)
  plt.imshow(batch_images[img_index]) # Display the original size image from the batch

  plt.title(labels[int(batch_labels[img_index])])
  plt.title(f"#{i} {index} - Label:"+labels[int(batch_labels[img_index])])
  plt.axis('off')

# Adjust subplot spacing
plt.tight_layout()
plt.subplots_adjust(wspace=.1, hspace=.3) # Adjust wspace (width space) and hspace (height space) as needed


plt.show()


## Modeling

In [ ]:
import tensorflow as tf


In [ ]:
model = tf.keras.models.Sequential([

    #Extraction de caractéristiques
    tf.keras.layers.Conv2D(filters=32, kernel_size=(3, 3), padding="same", activation='relu', input_shape=(224, 224,3)),
    tf.keras.layers.MaxPooling2D(2,2),
    tf.keras.layers.Conv2D(filters=64, kernel_size=(3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D(2,2),
    tf.keras.layers.Conv2D(filters=128, kernel_size=(3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D(2,2),
    tf.keras.layers.Conv2D(filters=128, kernel_size=(3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D(2,2),

    # applatir
    tf.keras.layers.Flatten(),

    # Dense
    tf.keras.layers.Dense(units=128, activation="relu"),
    tf.keras.layers.Dense(units=1, activation="sigmoid"),
])


In [ ]:
model.summary()


In [ ]:
model_ckp = tf.keras.callbacks.ModelCheckpoint(filepath="best_model.h5",
                            monitor="val_accuracy",
                            mode="max",
                            save_best_only=True)
stop = tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=7, restore_best_weights=True)


In [ ]:
model.compile(optimizer=tf.keras.optimizers.RMSprop(learning_rate=0.001),
              loss="binary_crossentropy",
              metrics=['accuracy'])


In [ ]:
h = model.fit(train_generator, epochs=50,
             validation_data=test_generator,
             callbacks = [model_ckp, stop])


In [ ]:
tf.keras.utils.plot_model(model)


In [ ]:
# Nombre d'itérations = Nombres de train / nombre batch_size=32 (Du ImageDataGenerator)
22564/32


In [ ]:
# model_save = tf.keras.models.load_model('best_model.h5')
# model_save.evaluate(train_generator, test_generator)


## Data Augmentation

In [ ]:
# avant augmentation
train_data_generator = ImageDataGenerator(rescale=1/255.0)
test_data_generator = ImageDataGenerator(rescale=1/255.0)


# après augmentation
train_data_generator = ImageDataGenerator(rescale=1/255.0,
                                          rotation_range=40,
                                          width_shift_range=0.2,
                                          height_shift_range=0.2,
                                          shear_range=0.2,
                                          zoom_range=0.2,
                                          horizontal_flip=True,
                                          fill_mode="nearest")
test_data_generator = ImageDataGenerator(rescale=1/255.0)


In [ ]:
train_generator = train_data_generator.flow_from_directory(
    directory = train_dir,
    target_size = (224, 224),
    batch_size=32,
    class_mode="binary"

)


test_generator = test_data_generator.flow_from_directory(
    directory = test_dir,
    target_size = (224, 224),
    batch_size=32,
    class_mode="binary"

)


In [ ]:
model = tf.keras.models.Sequential([

    #Extraction de caractéristiques
    tf.keras.layers.Conv2D(filters=32, kernel_size=(3, 3), padding="same", activation='relu', input_shape=(224, 224,3)),
    tf.keras.layers.MaxPooling2D(2,2),
    tf.keras.layers.Conv2D(filters=64, kernel_size=(3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D(2,2),
    tf.keras.layers.Conv2D(filters=128, kernel_size=(3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D(2,2),
    tf.keras.layers.Conv2D(filters=128, kernel_size=(3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D(2,2),

    # applatir
    tf.keras.layers.Flatten(),

    # Dense
    tf.keras.layers.Dense(units=128, activation="relu"),
    tf.keras.layers.Dense(units=1, activation="sigmoid"),
])


In [ ]:
model_ckp = tf.keras.callbacks.ModelCheckpoint(filepath="aug_model.h5",
                            monitor="val_accuracy",
                            mode="max",
                            save_best_only=True)
stop = tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=6, restore_best_weights=True)
model.compile(optimizer=tf.keras.optimizers.RMSprop(learning_rate=0.001),
              loss="binary_crossentropy",
              metrics=['accuracy'])


In [ ]:
h = model.fit(train_generator, epochs=50,
             validation_data=test_generator,
             callbacks = [model_ckp, stop])


In [ ]:
loss: 0.2942 - accuracy: 0.8854 - val_loss: 0.2266 - val_accuracy: 0.9180
loss: 0.3518 - accuracy: 0.8522 - val_loss: 0.2764 - val_accuracy: 0.8957


## Transfert Learning


In [ ]:
model.summary()


## Extraction de caractéristiques

In [ ]:
from tensorflow.keras.applications import vgg16


In [ ]:
pretrained_model = vgg16.VGG16(include_top=True, weights='imagenet', input_shape=(224, 224, 3))


In [ ]:
pretrained_model.summary()


In [ ]:
pretrained_model = vgg16.VGG16(include_top=False, weights='imagenet', input_shape=(224, 224, 3))


In [ ]:
pretrained_model.summary()


In [ ]:
(32, 224, 224, 3) ==> (32, 7, 7, 512)


### API Sequential

In [ ]:
model = tf.keras.models.Sequential([

    #Extraction de caractéristiques
    tf.keras.layers.Conv2D(filters=32, kernel_size=(3, 3), padding="same", activation='relu', input_shape=(224, 224,3)),
    tf.keras.layers.MaxPooling2D(2,2),
    tf.keras.layers.Conv2D(filters=64, kernel_size=(3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D(2,2),
    tf.keras.layers.Conv2D(filters=128, kernel_size=(3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D(2,2),
    tf.keras.layers.Conv2D(filters=128, kernel_size=(3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D(2,2),

    # applatir
    tf.keras.layers.Flatten(),

    # Dense
    tf.keras.layers.Dense(units=128, activation="relu"),
    tf.keras.layers.Dense(units=1, activation="sigmoid"),
                ])


In [ ]:
model = tf.keras.models.Sequential()


In [ ]:
model.add(tf.keras.layers.Conv2D(filters=32, kernel_size=(3, 3), padding="same", activation='relu', input_shape=(224, 224,3)))


In [ ]:
model.summary()


In [ ]:
model.add(tf.keras.layers.MaxPooling2D(2,2))


In [ ]:
model.summary()


### Api functional

In [ ]:
inputs = tf.keras.Input(shape=(224, 224, 3))
x =  tf.keras.layers.Conv2D(filters=32, kernel_size=(3, 3))(inputs)
x = tf.keras.layers.MaxPooling2D(2,2)(x)
outputs = tf.keras.layers.Flatten()(x)


In [ ]:
model = tf.keras.Model(inputs, outputs)


### Utilisation des filtres de VGG 16

In [ ]:
pretrained_model.trainable = False


In [ ]:
flatten_layer = tf.keras.layers.Flatten()


In [ ]:
inputs = tf.keras.Input(shape=(224, 224, 3))
x = pretrained_model(inputs, training=False)
x = flatten_layer(x) # 7*7*512
x = tf.keras.layers.Dense(128, activation='relu')(x)
outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)


model = tf.keras.Model(inputs, outputs)


In [ ]:
model.summary()


In [ ]:
model_ckp = tf.keras.callbacks.ModelCheckpoint(filepath="aug_model.h5",
                            monitor="val_accuracy",
                            mode="max",
                            save_best_only=True)
stop = tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=6, restore_best_weights=True)
model.compile(optimizer=tf.keras.optimizers.RMSprop(learning_rate=0.001),
              loss="binary_crossentropy",
              metrics=['accuracy'])


In [ ]:
h = model.fit(train_generator, epochs=50,
             validation_data=test_generator,
             callbacks = [model_ckp, stop])


In [ ]:
# loss: 0.2942 - accuracy: 0.8854 - val_loss: 0.2266 - val_accuracy: 0.9180
# loss: 0.3518 - accuracy: 0.8522 - val_loss: 0.2764 - val_accuracy: 0.8957
# loss: 0.2165 - accuracy: 0.9160 - val_loss: 0.3172 - val_accuracy: 0.8926


## Fine Tuning

In [ ]:
pretrained_model.summary()


In [ ]:
pretrained_model.layers


In [ ]:
pretrained_model.trainable=True


In [ ]:
pretrained_model.layers[:-4]


In [ ]:
for layer in pretrained_model.layers[:-4]:
  layer.trainable = False


In [ ]:
inputs = tf.keras.Input(shape=(224, 224, 3))
x = pretrained_model(inputs, training=False)
x = flatten_layer(x) # 7*7*512
x = tf.keras.layers.Dense(128, activation='relu')(x)
outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)


model = tf.keras.Model(inputs, outputs)


In [ ]:
model.summary()


In [ ]:
model_ckp = tf.keras.callbacks.ModelCheckpoint(filepath="aug_model.h5",
                            monitor="val_accuracy",
                            mode="max",
                            save_best_only=True)
stop = tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=2, restore_best_weights=True)
model.compile(optimizer=tf.keras.optimizers.RMSprop(learning_rate=0.001),
              loss="binary_crossentropy",
              metrics=['accuracy'])


In [ ]:
h = model.fit(train_generator, epochs=50,
             validation_data=test_generator,
             callbacks = [model_ckp, stop])
